# SylloGym — GRPO Training (Multi-Turn)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eliot-gtn/syllogym/blob/main/notebooks/train_syllogym_grpo.ipynb)

Fine-tune **Qwen3-4B** (or Qwen3-1.7B) on **SylloGym Judge** using **true multi-turn GRPO** — the model generates one answer per turn, the environment responds with new facts, and the full episode is trained as a single flat token sequence (Wordle/Blackjack pattern).

**How it works:**
1. `rollout_func` runs a full episode: reset → generate → step → generate → … → done
2. All turns are flattened into a single `(prompt_ids, completion_ids)` sequence
3. `env_mask` marks which tokens are model-generated (1) vs environment messages (0)
4. Reward is computed from the live episode and passed via `extra_fields`

**Recommended runtime:** A100 (40 GB) or H100

## 1. Install

In [ ]:
import os, sys, subprocess, shutil

# CRITICAL: must be set before unsloth is imported anywhere.
# Disables torch.compile ONLY in unsloth's chunked loss kernel (not vLLM).
# DO NOT set TORCHDYNAMO_DISABLE=1 — vLLM requires torch.compile / aot_compile.
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

# Wipe unsloth's compiled cache so it regenerates with compile disabled.
# Without this, unsloth loads a pre-compiled UnslothGRPOTrainer.py that was
# built without UNSLOTH_COMPILE_DISABLE, causing TorchRuntimeError at step 1.
_cache = '/content/unsloth_compiled_cache'
if os.path.exists(_cache):
    shutil.rmtree(_cache)
    print(f'Wiped {_cache}')

from google.colab import userdata

GH_USERNAME = 'eliot-gtn'
REPO_NAME   = 'syllogym'
GH_TOKEN    = userdata.get('GH_TOKEN')

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def run(cmd):
    subprocess.check_call(cmd, shell=True)

if not os.path.exists(REPO_NAME):
    run(f'git clone https://{GH_TOKEN}@github.com/{GH_USERNAME}/{REPO_NAME}.git')
else:
    run(f'cd {REPO_NAME} && git pull')

# Unsloth with vLLM + TRL 0.29.1 for rollout_func support
run('pip install -q --upgrade uv && uv pip install -q "unsloth[huggingface]" vllm')
pip('--force-reinstall', '--no-deps', 'trl==0.29.1')  # rollout_func requires TRL >= 0.29
pip('openenv-core>=0.2.1', 'datasets>=2.19.0,<3.0.0',
    'huggingface_hub>=0.23.0', 'matplotlib', 'wandb')

envs_path = os.path.abspath(f'{REPO_NAME}/envs')
if envs_path not in sys.path:
    sys.path.insert(0, envs_path)

import importlib; importlib.invalidate_caches()

# Smoke test
from syllogym_env import JudgeEnv
_env = JudgeEnv(seed=42)
_obs = _env.reset()
print('JudgeEnv OK')
print(f'  task={_obs.task_name}  turns={_obs.total_layers}  q={_obs.question[:60]}')

import trl; print(f'TRL version: {trl.__version__}')
from trl import GRPOTrainer
import inspect
has_rollout_func = 'rollout_func' in inspect.signature(GRPOTrainer.__init__).parameters
print(f'rollout_func supported: {has_rollout_func}')
assert has_rollout_func, 'TRL 0.29+ required for rollout_func'
print('Installation complete.')

## 2. Configuration

In [ ]:
import os, sys, warnings, logging, json
import torch

warnings.filterwarnings('ignore', category=FutureWarning, module='transformers')
logging.getLogger('transformers').setLevel(logging.ERROR)

HF_USERNAME  = 'farffadet'  # TODO: set your HF username

MODEL_NAME   = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
MAX_SEQ_LEN  = 6144  # 6144 → max_ctx=3072, prevents episode truncation

MAX_STEPS     = 200
BATCH_SIZE    = 8
NUM_GEN       = 8
LEARNING_RATE = 2e-6
MAX_COMPLETION_LEN = 3072

EVAL_EPISODES = 5

LOG_FILE = 'training.log'
_file_handler = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
_file_handler.setFormatter(logging.Formatter('%(asctime)s %(message)s'))
logging.getLogger('syllogym_train').addHandler(_file_handler)
logging.getLogger('syllogym_train').setLevel(logging.INFO)

EPISODE_LOG_FILE = 'episodes.log'
_ep_handler = logging.FileHandler(EPISODE_LOG_FILE, mode='w', encoding='utf-8')
_ep_handler.setFormatter(logging.Formatter('%(message)s'))
logging.getLogger('syllogym_episodes').addHandler(_ep_handler)
logging.getLogger('syllogym_episodes').setLevel(logging.INFO)
logging.getLogger('syllogym_episodes').propagate = False

print(f'Config loaded: {MAX_STEPS} steps, batch={BATCH_SIZE}, num_gen={NUM_GEN}, lr={LEARNING_RATE}')
print(f'  model: {MODEL_NAME}')
print(f'  logs: {LOG_FILE} (training) | {EPISODE_LOG_FILE} (conversations)')


## 3. Load model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=32,
    gpu_memory_utilization=0.75,  # 0.6 → 0.75 on H100 80GB
    enforce_eager=True,          # Blackwell (B200/RTX PRO 6000) compat — disables torch.compile in vLLM
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=64,
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded.')

## 4. Helpers

In [ ]:
import re
from syllogym_env import JudgeEnv, JudgeObs

SYSTEM_PROMPT = """You are an expert judge. You will receive facts about a legal case one step at a time.
At each step, answer the question with exactly one of the valid answers listed (e.g. Yes or No).
Reason carefully — new information may require you to revise your previous ruling.

Think briefly (2-3 sentences max) before answering. Be concise.

Format your answer as:
<answer>Yes</answer>  or  <answer>No</answer>

Always end with the <answer> tag."""


def format_turn_prompt(obs: JudgeObs) -> str:
    """Format a single turn into a user message."""
    if obs.layer_index == 0:
        return (
            f"[APPLICABLE LAW]\n{obs.rule}\n\n"
            f"[CASE FACTS]\n{obs.facts}\n\n"
            f"[QUESTION — Turn {obs.layer_index + 1}/{obs.total_layers}]\n"
            f"{obs.question}\n"
            f"Valid answers: {' | '.join(obs.valid_answers)}"
        )
    parts = []
    if obs.new_info:
        tag = "⚠ CORRECTION" if obs.is_twist else "NEW INFORMATION"
        parts.append(f"[{tag}]\n{obs.new_info}")
    parts.append(
        f"[QUESTION — Turn {obs.layer_index + 1}/{obs.total_layers}]\n"
        f"{obs.question}\n"
        f"Valid answers: {' | '.join(obs.valid_answers)}"
    )
    return "\n\n".join(parts)


def parse_answer(text: str) -> str | None:
    """Extract the LAST <answer>...</answer> tag from generated text."""
    matches = re.findall(r'<answer>\s*(.+?)\s*</answer>', text, re.IGNORECASE | re.DOTALL)
    return matches[-1].strip() if matches else None


print('Helpers defined.')

## 5. Rollout function (the core)

This is the heart of true multi-turn GRPO. For each episode:
1. Reset the env → get turn 0 observation
2. Build the initial prompt (system + turn 0 question)
3. Generate an answer via `_generate_single_turn`
4. Step the env with that answer → get new facts + next question  
5. Append new turn to conversation history, repeat
6. Flatten all turns into a single `(prompt_ids, completion_ids)` sequence

The `env_mask` marks which completion tokens are model-generated (1) vs injected environment messages (0), so the GRPO loss only trains on model tokens.

The final episode reward is passed via `extra_fields` and recovered by the reward function through `**kwargs`.

In [ ]:
from trl import GRPOTrainer
import inspect as _inspect
import re as _re
import logging as _logging

# MAX_PROMPT_LEN: fixed prompt length for padding.
# prompt_ids must be fixed-length so torch.compile sees concrete shapes.
# We use a fully-padded prompt_ids (no real content) and put the entire
# episode — env tokens + model tokens — into completion_ids, using
# env_mask to distinguish them. This gives TRL correct conditioning:
#   comp_turn_1 is conditioned on [system+user_0+comp_0+user_1],
#   exactly as it was during generation.
MAX_PROMPT_LEN = 1024  # MAX_SEQ_LEN(6144) - MAX_COMPLETION_LEN(3072) → 3072 context

_train_log = _logging.getLogger('syllogym_train')


def _log(msg: str):
    """Log to file + stdout."""
    _train_log.info(msg)

_ep_log = _logging.getLogger('syllogym_episodes')

def _log_episode_conversation(task_name: str, seed: int, messages_history: list[dict],
                               turn_rewards: list[float], episode_reward: float, skipped: bool = False):
    """Log the full conversation of an episode to episodes.log (silent — file only)."""
    sep = '─' * 60
    lines = [
        f'\n{sep}',
        f'EPISODE  task={task_name}  seed={seed}  reward={episode_reward:.2f}  turns={len(turn_rewards)}' +
        ('  [SKIPPED]' if skipped else f'  turn_rewards={[round(r,1) for r in turn_rewards]}'),
        sep,
    ]
    for msg in messages_history:
        role = msg['role'].upper()
        content = msg['content']
        lines.append(f'[{role}]')
        lines.append(content)
        lines.append('')
    _ep_log.info('\n'.join(lines))


def _make_dummy(pad_id: int, task_name: str) -> dict:
    """Return an all-padding dummy episode (reward=0, env_mask=0).
    Used when an episode is skipped (truncation, bad reset, etc.).
    env_mask=0 everywhere → advantage=0 → no gradient contribution.
    """
    return {
        "prompt_ids": [pad_id] * MAX_PROMPT_LEN,
        "completion_ids": [pad_id] * MAX_COMPLETION_LEN,
        "logprobs": [0.0] * MAX_COMPLETION_LEN,
        "env_mask": [0] * MAX_COMPLETION_LEN,
        "episode_reward": 0.0,
        "turn_rewards": [0.0],
        "completion_texts": [""],
        "task_name": task_name,
    }


def _tokenize_messages(messages: list[dict], tokenizer) -> list[int]:
    """Apply chat template and return token IDs as a plain list."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    return tokenizer.encode(text, add_special_tokens=False)


def _tokenize_messages_no_gen(messages: list[dict], tokenizer) -> list[int]:
    """Apply chat template WITHOUT add_generation_prompt (for env turns)."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return tokenizer.encode(text, add_special_tokens=False)


def _detect_generate_signature(trainer) -> str:
    try:
        sig = _inspect.signature(trainer._generate_single_turn)
        if 'multimodal_fields' not in sig.parameters:
            return 'strings'
    except (ValueError, TypeError):
        pass
    return 'token_ids'


def _call_generate(trainer, prompt_ids: list[int], prompt_text: str, sig: str):
    if sig == 'token_ids':
        comp_batch, lps_batch, _ = trainer._generate_single_turn(
            [prompt_ids], images=None, multimodal_fields={}
        )
    else:
        comp_batch, lps_batch, _ = trainer._generate_single_turn(
            [prompt_text], images=None
        )
    comp_ids = comp_batch[0]
    raw_lps = lps_batch[0] if lps_batch is not None else [0.0] * len(comp_ids)
    comp_lps = [-1.0 if (v != v or v == float('inf') or v == float('-inf')) else v for v in raw_lps]  # -1.0 avoids e^15 IS ratio spike from 0.0 (prob=1.0)
    return comp_ids, comp_lps


def _parse_prompt_row(prompt_messages: list[dict], fallback_tasks: list[str]) -> tuple[str, int]:
    import random as _rnd
    if prompt_messages and prompt_messages[0].get('role') == 'system':
        content = prompt_messages[0].get('content', '')
        m = _re.match(r'__task_name__=(.+)\|__seed__=(\d+)', content)
        if m:
            return m.group(1), int(m.group(2))
    return _rnd.choice(fallback_tasks), _rnd.randint(0, 100_000)


def rollout_episode(task_name: str, seed: int, trainer, gen_sig: str = None) -> dict:
    """
    True multi-turn rollout with correct TRL conditioning.

    Structure sent to TRL:
      prompt_ids:     [pad ... pad]                    ← fixed MAX_PROMPT_LEN, all padding
      completion_ids: [sys+user_0] [comp_0] [user_1] [comp_1] [user_2] [comp_2] ...
      env_mask:       [0  ...  0 ] [1 1 1 ] [0 0 0 ] [1 1 1 ] [0 0 0 ] [1 1 1 ] ...

    This ensures TRL computes log-prob of comp_turn_N conditioned on
    [sys+user_0+comp_0+user_1+...+user_N], exactly matching generation context.
    env_mask=0 on env tokens means they contribute to conditioning but not to the loss.

    If the episode context exceeds MAX_SEQ_LEN - MAX_COMPLETION_LEN at any turn,
    the episode is SKIPPED (returns a dummy with reward=0, env_mask=0) to prevent
    IS ratio explosion from truncated logprobs corrupting the group advantages.
    """
    tokenizer = trainer.processing_class
    if gen_sig is None:
        gen_sig = _detect_generate_signature(trainer)

    pad_id = tokenizer.eos_token_id
    max_ctx = MAX_SEQ_LEN - MAX_COMPLETION_LEN  # max tokens for context (3072 with 6144/3072)

    env = JudgeEnv(seed=seed if seed >= 0 else None)
    obs = env.reset(task_name=task_name)
    if obs.done:
        _log(f'[SKIP] task={task_name} seed={seed} | done at reset')
        return _make_dummy(pad_id, task_name)

    # ── Fixed all-padding prompt (satisfies torch.compile shape constraint) ──
    fixed_prompt_ids = [pad_id] * MAX_PROMPT_LEN

    # ── Episode state ─────────────────────────────────────────────────────────
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": format_turn_prompt(obs)})

    # completion_ids accumulates: env_tokens + comp_tokens + env_tokens + ...
    all_completion_ids: list[int] = []
    all_logprobs: list[float] = []
    all_env_mask: list[int] = []
    turn_rewards: list[float] = []
    completion_texts: list[str] = []

    # Tokenize initial env context (system + first user turn) → env_mask=0
    initial_env_ids = _tokenize_messages(messages, tokenizer)

    # Check if even the initial context exceeds max_ctx — skip immediately
    if len(initial_env_ids) > max_ctx:
        _log(f'[SKIP] task={task_name} seed={seed} | initial ctx too long: {len(initial_env_ids)} > {max_ctx}')
        print(f'[SKIP] task={task_name} seed={seed} | initial ctx {len(initial_env_ids)} > {max_ctx}')
        return _make_dummy(pad_id, task_name)

    all_completion_ids.extend(initial_env_ids)
    all_logprobs.extend([0.0] * len(initial_env_ids))
    all_env_mask.extend([0] * len(initial_env_ids))

    turn_idx = 0
    while not obs.done:
        # Skip episode if context exceeds window — truncated logprobs cause IS ratio explosion
        if len(all_completion_ids) > max_ctx:
            _log(
                f'[SKIP] task={task_name} seed={seed} turn={turn_idx} | '
                f'ctx too long: {len(all_completion_ids)} > {max_ctx} tokens'
            )
            print(f'[SKIP] task={task_name} seed={seed} | ctx {len(all_completion_ids)} > {max_ctx} at turn {turn_idx}')
            _log_episode_conversation(task_name, seed, messages, turn_rewards, 0.0, skipped=True)
            return _make_dummy(pad_id, task_name)

        context_ids = all_completion_ids
        context_text = tokenizer.decode(context_ids, skip_special_tokens=False)

        comp_ids, comp_lps = _call_generate(trainer, context_ids, context_text, gen_sig)
        completion_text = tokenizer.decode(comp_ids, skip_special_tokens=True)
        answer = parse_answer(completion_text) or "__invalid__"

        nan_lps = sum(1 for v in comp_lps if v == -1.0)

        # Model tokens → env_mask=1
        all_completion_ids.extend(comp_ids)
        all_logprobs.extend(comp_lps)
        all_env_mask.extend([1] * len(comp_ids))
        completion_texts.append(completion_text)

        obs = env.step(answer)
        turn_correct = 1.0 if (obs.reward is not None and obs.reward >= 0.5) else 0.0
        turn_rewards.append(turn_correct)

        # ── Per-turn log ──────────────────────────────────────────────────────
        _log(
            f'[TURN] task={task_name} seed={seed} turn={turn_idx} '
            f'answer={answer!r} correct={int(turn_correct)} '
            f'comp_tokens={len(comp_ids)} nan_lps={nan_lps}'
            + (f'\n  text: {completion_text[:300].replace(chr(10), " ")}' if answer == "__invalid__" else '')
        )

        if not obs.done:
            next_user_content = format_turn_prompt(obs)

            # Build env segment directly — no full re-tokenization, no diffing.
            # Special tokens act as natural BPE boundaries, so tokenizing the
            # inter-turn segment in isolation is safe and always correct.
            segment = (
                f"<|im_end|>\n"
                f"<|im_start|>user\n"
                f"{next_user_content}"
                f"<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            new_env_ids = tokenizer.encode(segment, add_special_tokens=False)

            # Update messages for context tracking
            messages.append({"role": "assistant", "content": completion_text})
            messages.append({"role": "user", "content": next_user_content})

            all_completion_ids.extend(new_env_ids)
            all_logprobs.extend([0.0] * len(new_env_ids))
            all_env_mask.extend([0] * len(new_env_ids))

        turn_idx += 1

    # Append final assistant turn to messages (only non-final turns are appended in the loop)
    if completion_texts:
        messages.append({"role": "assistant", "content": completion_texts[-1]})

    episode_reward = env.reward
    _log(
        f'[EPISODE] task={task_name} seed={seed} | '
        f'turns={len(turn_rewards)} reward={episode_reward:.2f} '
        f'turn_rewards={[round(r,1) for r in turn_rewards]} '
        f'total_tokens={len(all_completion_ids)}'
    )
    _log_episode_conversation(task_name, seed, messages, turn_rewards, episode_reward)

    return {
        "prompt_ids": fixed_prompt_ids,
        "completion_ids": all_completion_ids,
        "logprobs": all_logprobs,
        "env_mask": all_env_mask,
        "episode_reward": episode_reward,
        "turn_rewards": turn_rewards,
        "completion_texts": completion_texts,
        "task_name": task_name,
    }


def make_rollout_func(tasks: list[str]):
    """
    Returns a rollout_func compatible with TRL 0.29 GRPOTrainer.

    prompt_ids: fixed all-padding (MAX_PROMPT_LEN) — satisfies torch.compile
    completion_ids: full episode interleaved [env|comp|env|comp|...]
    env_mask: 0 for env tokens, 1 for model tokens
    Right-padded to MAX_COMPLETION_LEN.
    """
    _gen_sig = None
    _batch_count = [0]

    def rollout_func(prompts: list, trainer) -> dict:
        nonlocal _gen_sig
        if _gen_sig is None:
            _gen_sig = _detect_generate_signature(trainer)
            print(f'[rollout_func] _generate_single_turn signature: {_gen_sig}')

        pad_id = trainer.processing_class.eos_token_id
        all_prompt_ids, all_completion_ids = [], []
        all_logprobs, all_env_mask = [], []
        all_episode_rewards, all_turn_rewards = [], []
        all_completion_texts, all_tasks = [], []

        for prompt_messages in prompts:
            if not isinstance(prompt_messages, list):
                prompt_messages = [{"role": "user", "content": str(prompt_messages)}]
            task_name, seed = _parse_prompt_row(prompt_messages, tasks)
            result = rollout_episode(task_name, seed, trainer, gen_sig=_gen_sig)

            # Right-pad completion_ids to MAX_COMPLETION_LEN
            comp = result["completion_ids"][:MAX_COMPLETION_LEN]
            lps  = result["logprobs"][:MAX_COMPLETION_LEN]
            mask = result["env_mask"][:MAX_COMPLETION_LEN]
            pad_c = MAX_COMPLETION_LEN - len(comp)
            comp += [pad_id] * pad_c
            lps  += [0.0]    * pad_c
            mask += [0]      * pad_c

            # prompt_ids is already fixed MAX_PROMPT_LEN (all padding)
            prompt = result["prompt_ids"]  # already MAX_PROMPT_LEN

            all_prompt_ids.append(prompt)
            all_completion_ids.append(comp)
            all_logprobs.append(lps)
            all_env_mask.append(mask)
            all_episode_rewards.append(result["episode_reward"])
            all_turn_rewards.append(result["turn_rewards"])
            all_completion_texts.append(result["completion_texts"])
            all_tasks.append(result["task_name"])

        # ── Patch dummy rewards to group mean (avoid biasing advantages) ──────
        # Dummies have env_mask=all-zeros. Give them reward=mean of valid episodes
        # so their advantage = 0 → no gradient contribution, but group mean unchanged.
        is_dummy = [all(m == 0 for m in mask) for mask in all_env_mask]
        valid_rewards = [r for r, d in zip(all_episode_rewards, is_dummy) if not d]
        mean_valid_r = sum(valid_rewards) / len(valid_rewards) if valid_rewards else 0.0
        skip_count = sum(is_dummy)
        for i, d in enumerate(is_dummy):
            if d:
                all_episode_rewards[i] = mean_valid_r

        # ── Batch summary log ─────────────────────────────────────────────────
        _batch_count[0] += 1
        rewards = all_episode_rewards
        mean_r = sum(rewards) / len(rewards) if rewards else 0.0
        advantages = [r - mean_r for r in rewards]
        max_adv = max(abs(a) for a in advantages) if len(rewards) > 1 else 0.0
        env_mask_ratio = sum(sum(m) for m in all_env_mask) / max(sum(len(m) for m in all_env_mask), 1)
        nan_lp_count = sum(1 for lps in all_logprobs for v in lps if v == -1.0)
        zero_reward_frac = sum(1 for r in rewards if r == 0.0) / len(rewards) if rewards else 0.0
        _log(
            f'[BATCH] batch={_batch_count[0]} n={len(rewards)} '
            f'mean_r={mean_r:.3f} max_adv={max_adv:.2f} '
            f'zero_frac={zero_reward_frac:.2f} nan_lps={nan_lp_count} '
            f'env_mask_ratio={env_mask_ratio:.3f} '
            f'tasks={all_tasks}'
        )

        # ── Diagnostic log for aberrant batches ──────────────────────────────
        if max_adv > 3.0 or nan_lp_count > 0:
            print(f'[DIAG] max_adv={max_adv:.2f} | nan_lps={nan_lp_count} | tasks={all_tasks} | rewards={[round(r,2) for r in rewards]} | env_mask_ratio={env_mask_ratio:.3f}')

        return {
            "prompt_ids": all_prompt_ids,
            "completion_ids": all_completion_ids,
            "logprobs": all_logprobs,
            "env_mask": all_env_mask,
            "episode_reward": all_episode_rewards,
            "turn_rewards": all_turn_rewards,
            "completion_texts": all_completion_texts,
            "task_name": all_tasks,
        }

    return rollout_func


print('rollout_func defined.')


## 6. Build task-sampler dataset

The dataset is minimal — just `{task_name, seed}` rows. The actual episode content is
generated live inside `rollout_func`. The dataset only controls which tasks get sampled.

In [ ]:
from datasets import Dataset
import random as _rnd

# Large static dataset with random seeds in [0, 1_000_000].
# Effectively infinite diversity — model won't see the same episode twice.
TASKS = [
    'diversity_4', 'diversity_5',
    'miranda_4', 'miranda_5',
    'consideration_2', 'consideration_3', 'consideration_4',
    'terry_3', 'terry_4',
    'sof_4',
    'hearsay_3', 'hearsay_4',
    'adverse_possession_4', 'adverse_possession_5',
]

TASK_WEIGHTS = [int(task.split('_')[-1]) for task in TASKS]
DATASET_SIZE = 4000

_rng = _rnd.Random()  # unseeded → different each run
rows = []
for _ in range(DATASET_SIZE):
    task = _rng.choices(TASKS, weights=TASK_WEIGHTS, k=1)[0]
    seed = _rng.randint(0, 1_000_000)
    rows.append({
        "prompt": [
            {
                "role": "system",
                "content": f"__task_name__={task}|__seed__={seed}",
            }
        ],
    })

train_dataset = Dataset.from_list(rows)
print(f'Dataset: {len(train_dataset)} rows, {len(TASKS)} tasks, random seeds in [0, 1M]')


## 7. Reward functions (3 independent signals)

With `rollout_func`, rewards are computed **during rollout** (live `env.step`).
Three independent reward functions reduce `frac_reward_zero_std` by providing
different variance profiles — the model can get partial credit even on failed episodes.

| Function | Signal | Range | Purpose |
|---|---|---|---|
| `reward_format` | `<answer>Yes/No</answer>` present | 0–0.3 | Always non-zero variance; teaches format |
| `reward_turn_acc` | Mean per-turn correctness | 0–1.0 | Dense signal even on partial episodes |
| `reward_episode` | All turns correct | 0 or 1.0 | Sparse final signal; high variance |

In [ ]:
import logging
import re, json

_step_count = 0
_ANSWER_RE = re.compile(r'<answer>\s*(yes|no)\s*</answer>', re.IGNORECASE)


def reward_format(completions, completion_texts=None, **kwargs) -> list[float]:
    """
    0.3 if every turn's completion contains a valid <answer>Yes/No</answer> tag.
    0.15 if at least one turn has the tag (partial credit).
    0.0 if no tag found anywhere.

    Uses completion_texts (list[list[str]] per episode) from rollout extra_fields.
    Falls back to raw completions strings if not available.
    """
    rewards = []
    if completion_texts is not None:
        for ep_texts in completion_texts:
            if not ep_texts:
                rewards.append(0.0)
                continue
            hits = sum(1 for t in ep_texts if _ANSWER_RE.search(t))
            if hits == len(ep_texts):
                rewards.append(0.3)
            elif hits > 0:
                rewards.append(0.15)
            else:
                rewards.append(0.0)
    else:
        # Fallback: single completion string per row
        for c in completions:
            text = c[0].get('content', '') if isinstance(c, list) else str(c)
            rewards.append(0.3 if _ANSWER_RE.search(text) else 0.0)
    return rewards


def reward_turn_acc(completions, turn_rewards=None, **kwargs) -> list[float]:
    """
    Mean per-turn accuracy across all turns attempted in the episode.
    Range: 0.0–1.0.  Dense signal — always varies even on partially-correct episodes.
    """
    if turn_rewards is None:
        return [0.0] * len(completions)
    rewards = []
    for ep_turns in turn_rewards:
        if not ep_turns:
            rewards.append(0.0)
        else:
            rewards.append(sum(ep_turns) / len(ep_turns))
    return rewards


def reward_episode(completions, episode_reward=None, task_name=None, turn_rewards=None, **kwargs) -> list[float]:
    """
    1.0 if all turns correct (episode_reward >= 0.5), 0.0 otherwise.
    Logs training progress every 5 steps.
    """
    global _step_count
    _step_count += 1

    if episode_reward is None:
        return [0.0] * len(completions)

    if isinstance(episode_reward, (int, float)):
        rewards = [float(episode_reward)] * len(completions)
    else:
        rewards = [float(r) for r in episode_reward]

    mean_r = sum(rewards) / len(rewards) if rewards else 0.0
    task_str = str(task_name[0] if isinstance(task_name, list) else task_name)

    # Compute mean turn acc for logging
    mean_turn = 0.0
    if turn_rewards is not None:
        flat = [t for ep in turn_rewards for t in ep]
        mean_turn = sum(flat) / len(flat) if flat else 0.0

    if _step_count % 5 == 1 or mean_r > 0:
        print(
            f'[step {_step_count:>4d} | {task_str:<22s}]  '
            f'ep={mean_r:.2f}  turn_acc={mean_turn:.2f}  '
            f'rewards={[f"{r:.2f}" for r in rewards[:4]]}'
        )
    logging.getLogger("syllogym_train").info(
        f'step={_step_count} task={task_str} ep={mean_r:.3f} '
        f'turn_acc={mean_turn:.3f} rewards={rewards}'
    )
    return rewards


def _patch_trainer_log(trainer):
    _orig_log = trainer.log
    def _log_with_file(metrics, *args, **kwargs):
        _orig_log(metrics, *args, **kwargs)
        try:
            logging.getLogger("syllogym_train").info('METRICS ' + json.dumps(
                {k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()}
            ))
        except Exception:
            pass
    trainer.log = _log_with_file


print('Reward functions defined (format + turn_acc + episode).')

## 8. Debug: single episode rollout (before training)

Dry-run a full episode to verify the rollout works end-to-end:
real `env.step`, live answer parsing, token accumulation.

In [ ]:
import torch

FastLanguageModel.for_inference(model)


class _FakeTrainer:
    """Minimal trainer stub for eval/dry-run outside the training loop."""
    processing_class = tokenizer

    def _generate_single_turn(self, prompt_ids, images, multimodal_fields):
        """Direct greedy generation using the loaded model (not vLLM).

        max_new_tokens matches MAX_COMPLETION_LEN used during training so
        eval conditions are comparable to what the model saw at train time.
        Greedy (do_sample=False) for deterministic, reproducible eval.
        """
        device = next(model.parameters()).device
        results_ids = []
        results_lps = []
        for ids in prompt_ids:
            input_ids = torch.tensor([ids], device=device)
            with torch.no_grad():
                out = model.generate(
                    input_ids,
                    max_new_tokens=MAX_COMPLETION_LEN,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            comp = out[0][len(ids):].tolist()
            results_ids.append(comp)
            results_lps.append([0.0] * len(comp))
        return results_ids, results_lps, {}


TASK = 'diversity_3'
SEED = 99

print(f'=== Debug rollout: {TASK} (seed={SEED}) ===')
result = rollout_episode(TASK, SEED, _FakeTrainer())
print(f'Turns completed — reward: {result["episode_reward"]:.2f}')
print(f'Total prompt tokens:     {len(result["prompt_ids"])}')
print(f'Total completion tokens: {len(result["completion_ids"])}')
print(f'env_mask sum (model toks): {sum(result["env_mask"])} / {len(result["env_mask"])}')

# Decode completions turn by turn
print('\n--- Completion text ---')
print(tokenizer.decode(result['completion_ids'], skip_special_tokens=True)[:500])

## 9. GRPO Training

True multi-turn GRPO with `rollout_func`:
- `rollout_func` drives generation (no `reward_funcs` doing env interaction)
- `env_mask` ensures loss only on model tokens, not env messages
- One reward function reads pre-computed rewards from `extra_fields`

In [ ]:
import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))
wandb.init(project="syllogym", name=f"grpo-{MAX_STEPS}steps-{BATCH_SIZE}bs-{NUM_GEN}gen")
print("wandb initialized.")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
from huggingface_hub import HfApi as _HfApi
from google.colab import userdata as _userdata

HF_TOKEN = _userdata.get('HF_TOKEN')
HF_REPO_CHECKPOINTS = f'{HF_USERNAME}/syllogym-judge-qwen3-4b-grpo-checkpoints'

class HFCheckpointCallback(TrainerCallback):
    """Push LoRA adapter to HF Hub after every save_steps.
    Uses upload_folder (fast, no merge) — adapter is ~200MB not 8GB.
    """
    def __init__(self, repo_id: str, token: str, output_dir: str):
        self.api = _HfApi(token=token)
        self.repo_id = repo_id
        self.output_dir = output_dir
        # Create repo once (no-op if already exists)
        self.api.create_repo(repo_id, repo_type='model', exist_ok=True)
        print(f'[HFCheckpointCallback] will push to {repo_id}')

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        path_in_repo = f'checkpoint-{step}'
        local_path = f'{self.output_dir}/checkpoint-{step}'

        # ── Push LoRA checkpoint to HF ────────────────────────────────────────
        try:
            self.api.upload_folder(
                repo_id=self.repo_id,
                folder_path=local_path,
                path_in_repo=path_in_repo,
                repo_type='model',
                commit_message=f'checkpoint step {step}',
            )
            print(f'[HF] pushed checkpoint-{step} → {self.repo_id}')
            _log(f'[CHECKPOINT] step={step} pushed to {self.repo_id}/{path_in_repo}')
        except Exception as e:
            print(f'[HF] push failed at step {step}: {e}')

        # ── Copy episodes.log to Google Drive (overwrite) ───────────────────────
        try:
            import shutil as _shutil
            gdrive_dir = '/content/drive/MyDrive/syllogym_logs'
            os.makedirs(gdrive_dir, exist_ok=True)
            _shutil.copy2(EPISODE_LOG_FILE, f'{gdrive_dir}/episodes.log')
            _shutil.copy2(LOG_FILE, f'{gdrive_dir}/training.log')
            print(f'[GDrive] logs saved → {gdrive_dir}/')
        except Exception as e:
            print(f'[GDrive] log save failed: {e}')


# ── Mount Google Drive (for log backup) ──────────────────────────────────────
try:
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=False)
    print('[GDrive] mounted at /content/drive')
except Exception as e:
    print(f'[GDrive] mount failed (logs will not be saved to Drive): {e}')

FastLanguageModel.for_training(model)

training_args = GRPOConfig(
    use_vllm=True,
    temperature=0.9,            # 0.9: less exploration variance, more stable rollouts
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=5,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_generations=NUM_GEN,
    max_completion_length=MAX_COMPLETION_LEN,
    mask_truncated_completions=True,
    max_steps=MAX_STEPS,
    save_steps=20,              # push to HF every 20 steps (~40min on A100)
    max_grad_norm=1.0,          # 0.5 was clipping useful signal (grad 2-10 → 6% of update)
    report_to='wandb',
    run_name=f'syllogym-grpo-{MAX_STEPS}steps',
    output_dir='syllogym_grpo_checkpoints',
    epsilon=0.2,
    epsilon_high=0.28,        # DAPO clip-higher: more room for positive advantages
    beta=0.02,               # 0.02: stronger KL restraint, calibrated for 4B multi-turn
    loss_type="dr_grpo",     # replaces (R-mean)/std with R-mean → stable when std→0
    scale_rewards=False,     # CRITICAL with dr_grpo: disable TRL std-normalization
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_format, reward_turn_acc, reward_episode],
    args=training_args,
    train_dataset=train_dataset,
    rollout_func=make_rollout_func(TASKS),
    callbacks=[HFCheckpointCallback(HF_REPO_CHECKPOINTS, HF_TOKEN, training_args.output_dir)],
)

_patch_trainer_log(trainer)

print('Starting GRPO training (true multi-turn)...')
print(f'  Dataset: {len(train_dataset)} rows, random seeds in [0, 1M]')
print(f'  Steps: {MAX_STEPS}, batch={BATCH_SIZE}, num_gen={NUM_GEN}')
print(f'  Temperature: 0.9 | beta=0.02 | loss_type=dr_grpo | max_grad_norm=1.0 | scale_rewards=False')
print(f'  max_completion_length: {MAX_COMPLETION_LEN}, mask_truncated_completions=True')
print(f'  reward_funcs: reward_format + reward_turn_acc + reward_episode')
print(f'  log_file: {LOG_FILE}')
print(f'  HF checkpoints: {HF_REPO_CHECKPOINTS} (every 20 steps)')

_step_count = 0  # reset step counter in case cell is re-run
trainer.train()


## 10. Post-training evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Subset of tasks most representative of the training distribution.
# Skips the easiest (_2 with no twist) and focuses on mid/hard difficulty.
EVAL_TASKS = [
    'hearsay_2', 'hearsay_3', 'hearsay_4',
    'mens_rea_2', 'mens_rea_3',
    'diversity_3', 'diversity_5',
    'sof_3', 'sof_4',
    'miranda_3', 'miranda_5',
    'consideration_3', 'consideration_4',
    'adverse_possession_3', 'adverse_possession_5',
]
EVAL_EPISODES = 5  # per task — 5 * 15 * 2 passes ≈ 20-30min


def eval_task(task_name: str, label: str, n_episodes: int = EVAL_EPISODES) -> float:
    rewards = []
    for i in range(n_episodes):
        result = rollout_episode(task_name, 1000 + i, _FakeTrainer())
        rewards.append(result['episode_reward'])
        mean_so_far = float(np.mean(rewards))
        print(f'  [{label}] {task_name} [{i+1}/{n_episodes}]  '
              f'ep={result["episode_reward"]:.2f}  mean={mean_so_far:.3f}')
    mean = float(np.mean(rewards)) if rewards else 0.0
    print(f'  → {task_name}: {mean:.3f}\n')
    return mean


# ── Pass 1: baseline (weights currently in memory) ───────────────────────────
FastLanguageModel.for_inference(model)
print('=== BASELINE ===\n')
base_scores = {}
for task in EVAL_TASKS:
    base_scores[task] = eval_task(task, 'base')

base_mean = float(np.mean(list(base_scores.values())))
print(f'Baseline mean: {base_mean:.3f}\n')

# ── Load checkpoint ───────────────────────────────────────────────────────────
CHECKPOINT = 'syllogym_grpo_checkpoints/checkpoint-100'
print(f'Loading {CHECKPOINT}...')
model.load_adapter(CHECKPOINT, adapter_name='default')
FastLanguageModel.for_inference(model)
print('Checkpoint loaded.\n')

# ── Pass 2: checkpoint ────────────────────────────────────────────────────────
print(f'=== CHECKPOINT ({CHECKPOINT}) ===\n')
trained_scores = {}
for task in EVAL_TASKS:
    trained_scores[task] = eval_task(task, 'ckpt')
    delta = trained_scores[task] - base_scores[task]
    print(f'  DELTA {task}: {delta:+.3f}\n')

trained_mean = float(np.mean(list(trained_scores.values())))

# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 52)
print(f'Baseline mean:   {base_mean:.3f}')
print(f'Checkpoint mean: {trained_mean:.3f}')
print(f'Overall delta:   {trained_mean - base_mean:+.3f}')
print('=' * 52)
for task in EVAL_TASKS:
    delta = trained_scores[task] - base_scores[task]
    bar = '█' * int(abs(delta) * 20)
    sign = '+' if delta >= 0 else '-'
    print(f'  {task:<32s} {sign}{abs(delta):.3f}  {bar}')

# ── Chart ─────────────────────────────────────────────────────────────────────
x = np.arange(len(EVAL_TASKS))
w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, [base_scores[t] for t in EVAL_TASKS],    w,
       label='Baseline',    color='#6c8ebf', alpha=0.85)
ax.bar(x + w/2, [trained_scores[t] for t in EVAL_TASKS], w,
       label=f'Checkpoint', color='#82b366', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(EVAL_TASKS, rotation=30, ha='right', fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Mean episode reward')
ax.set_title(
    f'SylloGym Judge — Baseline {base_mean:.3f} → Checkpoint {trained_mean:.3f}'
    f'  (Δ{trained_mean - base_mean:+.3f})'
)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('syllogym_eval_checkpoint.png', dpi=150)
plt.show()

## 11. Push to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

REPO_ID = f'{HF_USERNAME}/syllogym-judge-qwen3-4b-grpo'

model.save_pretrained('syllogym_judge_qwen3_4b')
tokenizer.save_pretrained('syllogym_judge_qwen3_4b')
model.push_to_hub(REPO_ID, token=True)
tokenizer.push_to_hub(REPO_ID, token=True)

HfApi().upload_file(
    path_or_fileobj='syllogym_eval_results.png',
    path_in_repo='syllogym_eval_results.png',
    repo_id=REPO_ID, repo_type='model',
)

print(f'Model pushed to: https://huggingface.co/{REPO_ID}')